In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time

# =========================
# INPUT
# =========================
Leagues = [

"germany/bundesliga-2023-2024",
"germany/bundesliga-2022-2023",
"germany/bundesliga-2021-2022",

    ]

BASE = "https://us.soccerway.com/"

# =========================
# SETUP
# =========================
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)

final_data = []

# =========================
# LOOP OVER LEAGUES
# =========================
for league_path in Leagues:

    print(f"\nProcessing: {league_path}")

    # Extract metadata
    league_name = league_path.split("/")[1].split("-")[0]
    season = "-".join(league_path.split("-")[-2:])

    url = BASE + league_path + "/results/"
    driver.get(url)

    handles = set()

    # =========================
    # LOAD ALL MATCHES
    # =========================
    while True:
        try:
            links = driver.find_elements(By.CLASS_NAME, "eventRowLink")

            for link in links:
                href = link.get_attribute("href")
                if href:
                    handles.add(href)

            print(f"Collected matches: {len(handles)}")

            button = wait.until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, 'button[data-testid="wcl-buttonLink"]')
            ))

            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", button)
            driver.execute_script("arguments[0].click();", button)

            prev_count = len(links)
            wait.until(lambda d: len(d.find_elements(By.CLASS_NAME, "eventRowLink")) > prev_count)

        except:
            print("All matches loaded.")
            break

    print(f"Total matches in season: {len(handles)}")

    # =========================
    # SCRAPE MATCHES
    # =========================
    handles_list = list(handles)
    
    final_handles=[]
    for handle in handles_list:
        result=handle.split('/')
        result[5]=result[5]+"/summary/stats/overall"
        result='/'.join(result)
        final_handles.append(result)
    for match_url in final_handles:
        try:
            driver.get(match_url)

            wait.until(EC.presence_of_element_located(
                (By.CLASS_NAME, "detailScore__wrapper")
            ))
            wait.until(EC.presence_of_element_located(
                (By.CSS_SELECTOR, ".wcl-bold_NZXv6.wcl-scores-simple-text-01_-OvnR.wcl-scores_Na715")
            ))
            time.sleep(1)

            # Keep your BeautifulSoup logic
            soup = BeautifulSoup(driver.page_source, "html.parser")

            score_wrapper = soup.find('div', class_='detailScore__wrapper')
            spans = score_wrapper.find_all('span')

            home_goals = int(spans[0].get_text(strip=True))
            away_goals = int(spans[2].get_text(strip=True))

            # Your original team extraction
            home_team = match_url.split('/')[4].split('-')[0]
            away_team = match_url.split('/')[5].split('-')[0]

            ss = ["Corner kicks","Expected goals (xG)","Ball possession","Shots off target",
                  "Total shots","Shots on target","Yellow cards","Red cards","Fouls"]

            home_res = []
            away_res = []

            all_states = soup.find_all(
                'span',
                class_='wcl-bold_NZXv6 wcl-scores-simple-text-01_-OvnR wcl-scores_Na715'
            )

            for s in ss:
                state = soup.find(
                    'span',
                    class_='wcl-bold_NZXv6 wcl-scores-simple-text-01_-OvnR wcl-scores_Na715',
                    string=s
                )

                if state in all_states:
                    indx = all_states.index(state)

                    home_res.append(all_states[indx - 1].get_text())
                    away_res.append(all_states[indx + 1].get_text())
                else:
                    home_res.append(0)
                    away_res.append(0)

            # Keep your original logic
            home_res.append(away_res[1])
            away_res.append(home_res[1])

            home_res.append(home_team)
            home_res.append(away_team)

            away_res.append(away_team)
            away_res.append(home_team)

            home_res.append(home_goals)
            home_res.append(away_goals)

            away_res.append(away_goals)
            away_res.append(home_goals)

            home_res.append("Home")
            away_res.append("Away")

            # Result
            if home_goals > away_goals:
                home_res.append("W")
                away_res.append("L")
            elif home_goals < away_goals:
                home_res.append("L")
                away_res.append("W")
            else:
                home_res.append("D")
                away_res.append("D")

            # =========================
            # ADD LEAGUE + SEASON
            # =========================
            home_res.insert(0, season)
            home_res.insert(0, league_name)

            away_res.insert(0, season)
            away_res.insert(0, league_name)

            final_data.append(home_res)
            final_data.append(away_res)

            print(final_data)
        except Exception as e:
            print("Error:", e)
            continue

# =========================
# DONE
# =========================
driver.quit()

print("\nDONE ✅")
print(f"Total rows: {len(final_data)}")

for row in final_data[:5]:
    print(row)